# 全体性能と公平性 — adversary strength

`attribute_adversary_weight` (λ) を 0.1 / 1 / 3 / 10 と変えた run について、
分類性能と属性ごとの公平性を同じ test split で比較する。

この notebook は、まず run artifact・設定・split を読み込む入口を持つ。
予測 cache と群別指標を生成した後に、ここへ結果の表示・作図・読み取りを追加する。
checkpoint は全 run で validation AUROC により選ばれているため、公平性指標で選んだ結果ではない。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import numpy as np
import pandas as pd
import rootutils
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import (  # noqa: E402
    STYLE_SHEET,
    local_data_path,
    run_dir,
    split_csv,
)
from analysis.common.predictions import cache_path, load_cache  # noqa: E402
from analysis.common.run_artifacts import (  # noqa: E402
    read_config,
    read_epoch_metrics,
    read_run_record,
    selected_checkpoint,
)

matplotlib.style.use(STYLE_SHEET)

## 対象 run と定数

λ=0.1 は既存の初期比較から取り込んだ invariant run、λ=0 は adversary 無しの baseline。
λ=1 / 3 / 10 はこの study で水準を振った run である。全て seed 43 のため、ここでは
効果の確定ではなく、属性が消え始める水準と性能・gap の形を探索する。

In [ ]:
STUDY = "adversary_strength"
DATASET = "chexpert"
SPLIT = "test"
SEED = 43

RUN_IDS = {
    "baseline": "20260922T063725Z-resnet-chexpert-s43-efcd",
    "lambda_0.1": "20260922T063725Z-resnet-chexpert-attribute-invariant-s43-546c",
    "lambda_1": "20260922T111111Z-resnet-chexpert-attribute-invariant-s43-f5cb",
    "lambda_3": "20260922T133206Z-resnet-chexpert-attribute-invariant-s43-2d1b",
    "lambda_10": "20260922T155241Z-resnet-chexpert-attribute-invariant-s43-78d1",
}
LAMBDA = {
    "baseline": 0.0,
    "lambda_0.1": 0.1,
    "lambda_1": 1.0,
    "lambda_3": 3.0,
    "lambda_10": 10.0,
}
LABEL = {
    "baseline": "ResNet (λ=0)",
    "lambda_0.1": "attribute-invariant (λ=0.1)",
    "lambda_1": "attribute-invariant (λ=1)",
    "lambda_3": "attribute-invariant (λ=3)",
    "lambda_10": "attribute-invariant (λ=10)",
}
ATTRIBUTES = ["sex", "race", "age_group_65", "ethnicity"]
FAIRNESS_METRICS = ["Eopp0", "Eopp1", "Eodds", "AUROC gap", "bACC gap"]
PERFORMANCE_METRICS = ["accuracy", "balanced_accuracy", "auroc", "cross_entropy"]
RESULTS = ROOT / "analysis" / STUDY / "results"
FIGURES = ROOT / "analysis" / STUDY / "figures"
CACHE = ROOT / "analysis" / STUDY / "cache"

## run artifact の読み込み

`run.json` を数値の正本、`config.yaml` を λ と checkpoint 選択条件の正本として読む。
学習時の epoch 推移が残る run では `metrics.csv` も読み、後で validation 推移や collapse の
兆候を確認できるようにしておく。

In [ ]:
run_dirs = {name: run_dir(STUDY, run_id) for name, run_id in RUN_IDS.items()}
records = {name: read_run_record(path) for name, path in run_dirs.items()}
configs = {name: read_config(path / "config.yaml") for name, path in run_dirs.items()}

run_overview = pd.DataFrame(
    [
        {
            "condition": name,
            "label": LABEL[name],
            "lambda": LAMBDA[name],
            "seed": record["seed"],
            "status": record["status"],
            "checkpoint": selected_checkpoint(run_dirs[name]).name,
            "checkpoint_monitor": record["checkpoints"][0]["monitor"],
            "best_val_auroc": record["checkpoints"][0]["best"]["score"],
            "attribute_adversary_weight": configs[name]
            .get("model", {})
            .get("loss_fn", {})
            .get("attribute_adversary_weight", 0.0),
            "gradient_scale": configs[name].get("model", {}).get("loss_fn", {}).get("gradient_scale", 0.0),
        }
        for name, record in records.items()
    ]
).sort_values("lambda")
run_overview

In [ ]:
epoch_metrics = {}
for name, path in run_dirs.items():
    metrics_path = path / "metrics" / "metrics.csv"
    if metrics_path.is_file():
        epoch_metrics[name] = pd.DataFrame(read_epoch_metrics(metrics_path))
    else:
        epoch_metrics[name] = pd.DataFrame()

fit_summary = pd.DataFrame(
    [{"condition": name, "lambda": LAMBDA[name], **record["result_summary"]} for name, record in records.items()]
).sort_values("lambda")

fit_summary[["condition", "lambda", "val/auroc", "val/bacc", "val/loss"]]

## test split と支持数の読み込み

公平性 gap は群の人数、とくに陽性数が小さい群の影響を受ける。
そのため指標を読む前に、同じ test split の属性列と群の支持数を確認する。
予測 cache はこの CSV の行順と照合してから使う。

In [ ]:
SPLIT_PATH = split_csv(DATASET, SPLIT)
test_split = pd.read_csv(SPLIT_PATH)

# run が記録した data path が container の絶対 path でも、local repo の data/ へ解決できることを確認する。
recorded_data_dir = configs["lambda_1"].get("data", {}).get("data_dir")
resolved_data_dir = local_data_path(recorded_data_dir) if recorded_data_dir else SPLIT_PATH.parent.parent

test_split.shape, SPLIT_PATH, resolved_data_dir

In [ ]:
def support_table(frame: pd.DataFrame, attribute: str) -> pd.DataFrame:
    """属性群ごとの全体数と陽性数を、fairness 指標の前に確認する。"""
    return (
        frame.groupby(attribute, dropna=False)["target"]
        .agg(n="size", n_positive="sum")
        .reset_index()
        .sort_values(attribute)
    )


# target / attribute の列名は、groups.py の実装と突き合わせてから公平性集計に使う。
available_columns = sorted(test_split.columns)
available_columns

## 全体性能の算出

全体性能は属性欠損による母集団の違いを持ち込まないため、test split の全行で測る。
`recall_0` と `precision_0` は陰性クラス（class 0）を陽性側として計算する。
5 条件は全て seed 43 の 1 本なので、ここでは平均や標準偏差に畳まず、λ に沿った値をそのまま読む。

In [ ]:
test_images = test_split["image"].to_numpy(dtype=str)
caches = {name: load_cache(cache_path(CACHE, run_id, SPLIT), images=test_images) for name, run_id in RUN_IDS.items()}


def overall_metrics(cached: dict[str, np.ndarray]) -> dict[str, float]:
    """1 run の test 全体について、今回比較する性能指標を返す。"""
    target = cached["target"]
    prediction = cached["predictions"]
    positive_probability = cached["probabilities"][:, 1]
    return {
        "auroc": roc_auc_score(target, positive_probability),
        "bacc": balanced_accuracy_score(target, prediction),
        "recall_0": recall_score(target, prediction, pos_label=0),
        "precision_0": precision_score(target, prediction, pos_label=0, zero_division=0),
        "f1": f1_score(target, prediction),
    }


performance = (
    pd.DataFrame(
        [
            {"condition": name, "label": LABEL[name], "lambda": LAMBDA[name], **overall_metrics(cached)}
            for name, cached in caches.items()
        ]
    )
    .sort_values("lambda")
    .reset_index(drop=True)
)
performance

### 表示用の比較表

AUROC は threshold-free、残りは選択済み checkpoint の hard prediction に依存する。
したがって、AUROC と hard 指標の順位がずれる場合もそのまま記録する。

In [ ]:
METRIC_COLUMNS = ["auroc", "bacc", "recall_0", "precision_0", "f1"]
performance[["label", "lambda", *METRIC_COLUMNS]].style.format({metric: "{:.4f}" for metric in METRIC_COLUMNS})

## 再利用する表の保存

この表は、後続の公平性比較やレポートから参照できるよう `results/` に保存する。

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
performance.to_csv(RESULTS / f"overall_performance_{SPLIT}.csv", index=False)
RESULTS / f"overall_performance_{SPLIT}.csv"

## subgroup 別性能

属性ごとに subgroup を分け、各 subgroup の AUROC、balanced accuracy、recall_0、recall_1 を見る。
`recall_0` は陰性クラスの再現率（specificity）、`recall_1` は陽性クラスの再現率（TPR）である。
race は support の小さいカテゴリが gap を支配しないよう、White / Asian / Black に限定する。

In [ ]:
ATTRIBUTE_LABELS = {
    "age_group_65": (test_split["age"] >= 65).map({False: "<65", True: ">=65"}),
    "sex": test_split["sex"].map({0: "Male", 1: "Female"}),
    "race": test_split["race"].map({0: "White", 2: "Asian", 3: "Black"}),
}
ATTRIBUTE_MISSING = {
    "age_group_65": test_split["age_missing"].astype(bool) | test_split["age"].isna(),
    "sex": test_split["sex_missing"].astype(bool),
    "race": test_split["race_missing"].astype(bool),
}

demographics = pd.DataFrame(ATTRIBUTE_LABELS).mask(pd.DataFrame(ATTRIBUTE_MISSING))
demographics["race"].value_counts(dropna=False)

In [ ]:
SUBGROUP_METRICS = ["auroc", "bacc", "recall_0", "recall_1"]


def subgroup_metrics(target: np.ndarray, probabilities: np.ndarray, predictions: np.ndarray) -> dict[str, float]:
    """1 subgroup の性能を返す。片方の class が無い AUROC は欠損にする。"""
    has_both_classes = np.unique(target).size == 2
    return {
        "n": len(target),
        "n_positive": int((target == 1).sum()),
        "auroc": roc_auc_score(target, probabilities) if has_both_classes else np.nan,
        "bacc": balanced_accuracy_score(target, predictions) if has_both_classes else np.nan,
        "recall_0": recall_score(target, predictions, pos_label=0, zero_division=0),
        "recall_1": recall_score(target, predictions, pos_label=1, zero_division=0),
    }


subgroup_rows = []
for condition, cached in caches.items():
    for attribute in ATTRIBUTES[:3]:
        valid = demographics[attribute].notna().to_numpy()
        for group in demographics.loc[valid, attribute].sort_values().unique():
            index = valid & (demographics[attribute].to_numpy() == group)
            metrics = subgroup_metrics(
                cached["target"][index],
                cached["probabilities"][index, 1],
                cached["predictions"][index],
            )
            subgroup_rows.append(
                {
                    "condition": condition,
                    "label": LABEL[condition],
                    "lambda": LAMBDA[condition],
                    "attribute": attribute,
                    "group": group,
                    **metrics,
                }
            )

subgroup_performance = pd.DataFrame(subgroup_rows).sort_values(["lambda", "attribute", "group"])
subgroup_performance.to_csv(RESULTS / f"subgroup_performance_{SPLIT}.csv", index=False)
subgroup_performance

## subgroup 間の gap

各属性について subgroup の max−min を取り、性能のばらつきを比較する。
`Eopp1` は recall_1（TPR）の gap、`Eopp0` は recall_0（TNR）の gap、`Eodds` はその平均とする。
AUROC と BAcc の gap も併記し、gap の変化がどの性能軸で起きたかを分けて読む。

In [ ]:
GAP_METRICS = ["auroc", "bacc", "recall_0", "recall_1"]

gap_rows = []
for (condition, attribute), part in subgroup_performance.groupby(["condition", "attribute"], sort=False):
    row = {
        "condition": condition,
        "label": LABEL[condition],
        "lambda": LAMBDA[condition],
        "attribute": attribute,
        "n_groups": len(part),
        "min_n": int(part["n"].min()),
        "min_n_positive": int(part["n_positive"].min()),
    }
    for metric in GAP_METRICS:
        row[f"{metric}_worst"] = part[metric].min()
        row[f"{metric}_best"] = part[metric].max()
        row[f"{metric}_gap"] = part[metric].max() - part[metric].min()
    row["Eopp1"] = row["recall_1_gap"]
    row["Eopp0"] = row["recall_0_gap"]
    row["Eodds"] = (row["Eopp0"] + row["Eopp1"]) / 2
    gap_rows.append(row)

fairness_gaps = pd.DataFrame(gap_rows).sort_values(["lambda", "attribute"])
fairness_gaps.to_csv(RESULTS / f"fairness_gaps_{SPLIT}.csv", index=False)
fairness_gaps[
    ["label", "lambda", "attribute", "auroc_gap", "bacc_gap", "Eopp0", "Eopp1", "Eodds", "min_n", "min_n_positive"]
]

## subgroup 性能の可視化

属性ごとに subgroup の性能曲線を分ける。属性間で値のスケールが違うため、属性を行、
指標を列にした panel とし、同じ属性内の subgroup の動きを比較しやすくする。
λ=0 も含むので、横軸は数値の対数軸ではなく条件の順序軸にする。

In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)
condition_order = [name for name, _ in sorted(LAMBDA.items(), key=lambda item: item[1])]
condition_labels = [f"λ={LAMBDA[name]:g}" for name in condition_order]
group_colors = {
    "<65": "#0173B2",
    ">=65": "#DE8F05",
    "Female": "#029E73",
    "Male": "#D55E00",
    "White": "#0173B2",
    "Asian": "#DE8F05",
    "Black": "#CC78BC",
}

figure, axes = plt.subplots(3, 4, figsize=(15, 10), constrained_layout=True)
for row, attribute in enumerate(["age_group_65", "sex", "race"]):
    attribute_rows = subgroup_performance[subgroup_performance["attribute"] == attribute]
    for column, metric in enumerate(SUBGROUP_METRICS):
        axis = axes[row, column]
        for group, part in attribute_rows.groupby("group", sort=False):
            part = part.set_index("condition").reindex(condition_order)
            axis.plot(
                range(len(condition_order)),
                part[metric],
                marker="o",
                linewidth=1.8,
                label=group,
                color=group_colors.get(group),
            )
        if row == 0:
            axis.set_title(metric)
        if column == 0:
            axis.set_ylabel(attribute)
        axis.set_xticks(range(len(condition_order)), condition_labels)
        axis.grid(axis="y", alpha=0.25)
        axis.set_ylim(0, 1)
        if row == 2:
            axis.set_xlabel("adversary strength λ")
    handles, labels = axes[row, -1].get_legend_handles_labels()
    axes[row, -1].legend(
        handles, labels, title=attribute, loc="upper right", fontsize=9, title_fontsize=9, frameon=True, framealpha=0.9
    )
figure.suptitle("Subgroup performance on test split", y=1.08)
figure.savefig(FIGURES / f"subgroup_performance_{SPLIT}.png", dpi=180, bbox_inches="tight")
plt.show()

## gap の可視化

gap は小さいほど subgroup 間の差が小さい。AUROC / BAcc の gap と、
classification parity に対応する Eopp0 / Eopp1 / Eodds を同じ図で追い、
どの属性で、どの種類の差が縮んだかを見る。

In [ ]:
gap_plot_metrics = {
    "auroc_gap": "AUROC gap",
    "bacc_gap": "BAcc gap",
    "Eopp0": "Eopp0",
    "Eopp1": "Eopp1",
    "Eodds": "Eodds",
}
gap_colors = {
    "auroc_gap": "#0173B2",
    "bacc_gap": "#DE8F05",
    "Eopp0": "#029E73",
    "Eopp1": "#D55E00",
    "Eodds": "#CC78BC",
}

figure, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True, constrained_layout=True)
for axis, attribute in zip(axes, ["age_group_65", "sex", "race"], strict=True):
    part = fairness_gaps[fairness_gaps["attribute"] == attribute].set_index("condition").reindex(condition_order)
    for metric, label in gap_plot_metrics.items():
        axis.plot(
            range(len(condition_order)),
            part[metric],
            marker="o",
            linewidth=1.8,
            label=label,
            color=gap_colors[metric],
        )
    axis.set_title(attribute)
    axis.set_xticks(range(len(condition_order)), condition_labels)
    axis.set_xlabel("adversary strength λ")
    axis.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("gap (lower is better)")
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.08))
figure.suptitle("Fairness gaps on test split", y=1.15)
figure.savefig(FIGURES / f"fairness_gaps_{SPLIT}.png", dpi=180, bbox_inches="tight")
plt.show()